# コスメマスター

---
1. 本分析の流れ
* ローカルでのRAGチャットの構築：手元のPCで動く簡単なRAGチャットボットを作る。
* クラウドへのデプロイとAPI化：Dockerでコンテナ化し、クラウドにデプロイする。FastAPIなどを用いてバックエンドAPIを構築し、フロントエンドから呼び出す構造を作る。
* 実践的な運用設計：CI/CDパイプラインを組み、コードを更新した自動でデプロイされる仕組みを作る。

2. 全体像
* LLM＆RAG：コスメデータをもとに賢い回答を生成する論理。
* Docker：作成した論理を、環境に依存せずどこでも動くようにパッケージングする技術。
* AWS：パッケージを24時間安定して動かし、世界中からアクセス可能にするクラウドインフラ。
* CI/CDパイプライン：コードを書き換えたら自動でAWSへ反映する仕組み。

---

## 使用するライブラリ

In [1]:
import pandas as pd
import os
from langchain_community.document_loaders import DataFrameLoader
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

/var/folders/t_/3s2g661d2nz05h53sk48b4qc0000gp/T/ipykernel_92955/642646670.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DataFrameLoader


## データの前処理

### 1. APIキーの設定

#### OpenAI APIキーの取得と設定手順
1. OpenAIのプラットフォームにアクセス：ブラウザで`OpenAI API Platform`と検索し、アカウントを作成（またはログイン）する。
2. クレジットカードの設定：APIを利用するには少額のチャージが必要なため、ここは目を瞑るしかない。
3. APIの発行
   * メニューバーの`API Key`を開き、`Create new secret key`をクリックし、適当な名前をつけて作成する。
   * `sk-`から始まる長い文字列全文をコピーする（画面閉じると二度と見る事ができない）。
   * 以下コードの`os.environ["OPENAI_API_KEY"] =`の後にコピーアンドペーストする。

In [ ]:
os.environ["OPENAI_API_KEY"] = "Your_Key"

### 2.データの取り込み

In [58]:
df = pd.read_csv("./cosmetics.csv")
df.head()

,Label,Brand,Name,Price,Rank,Ingredients,Combination,Dry,Normal,Oily,Sensitive
0,Moisturizer,LA MER,Crème de la Mer,175,4.1,"Algae (Seaweed) Extract, Mineral Oil, Petrolat...",1,1,1,1,1
1,Moisturizer,SK-II,Facial Treatment Essence,179,4.1,"Galactomyces Ferment Filtrate (Pitera), Butyle...",1,1,1,1,1
2,Moisturizer,DRUNK ELEPHANT,Protini™ Polypeptide Cream,68,4.4,"Water, Dicaprylyl Carbonate, Glycerin, Ceteary...",1,1,1,1,0
3,Moisturizer,LA MER,The Moisturizing Soft Cream,175,3.8,"Algae (Seaweed) Extract, Cyclopentasiloxane, P...",1,1,1,1,1
4,Moisturizer,IT COSMETICS,Your Skin But Better™ CC+™ Cream with SPF 50+,38,4.1,"Water, Snail Secretion Filtrate, Phenyl Trimet...",1,1,1,1,1


In [59]:
# 欠損値処理
df = df.dropna(subset=["Name", "Ingredients"])

### 3.オリジナル列(AIが読みやすい形)の作成

In [60]:
df["ai_context"] = (
    "ブランド: " + df["Brand"] + "\n" +
    "商品名: " + df["Name"] + " (" + df["Label"] + ")\n" +
    "成分: " + df["Ingredients"]
)

In [61]:
print(df["ai_context"][0])

ブランド: LA MER
商品名: Crème de la Mer (Moisturizer)
成分: Algae (Seaweed) Extract, Mineral Oil, Petrolatum, Glycerin, Isohexadecane, Microcrystalline Wax, Lanolin Alcohol, Citrus Aurantifolia (Lime) Extract, Sesamum Indicum (Sesame) Seed Oil, Eucalyptus Globulus (Eucalyptus) Leaf Oil, Sesamum Indicum (Sesame) Seed Powder, Medicago Sativa (Alfalfa) Seed Powder, Helianthus Annuus (Sunflower) Seedcake, Prunus Amygdalus Dulcis (Sweet Almond) Seed Meal, Sodium Gluconate, Copper Gluconate, Calcium Gluconate, Magnesium Gluconate, Zinc Gluconate, Magnesium Sulfate, Paraffin, Tocopheryl Succinate, Niacin, Water, Beta-Carotene, Decyl Oleate, Aluminum Distearate, Octyldodecanol, Citric Acid, Cyanocobalamin, Magnesium Stearate, Panthenol, Limonene, Geraniol, Linalool, Hydroxycitronellal, Citronellol, Benzyl Salicylate, Citral, Sodium Benzoate, Alcohol Denat., Fragrance.


## LangChain用に変換

DataFrameLoaderで、LangChain用に変換し、**`ai_context`** をメインの文章として読み込む。

In [62]:
loader = DataFrameLoader(df, page_content_column="ai_context")
documents = loader.load()

## データベース（FAISS）の構築

埋め込み（embedding）を行い、データベース（FAISS）に保存する。

In [63]:
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

## LLMの設定

* **`"gpt-3.5-turbo"`**
  * コストパフォーマンス: 上位モデル（GPT-4系等）と比較してAPIの利用料金が安価である。
  * レスポンス速度：生成スピードが速い。
* `temperature=0`
  * ランダム性の排除: 「回答の創造性（ばらつき）」を決める数値。０に設定することで、一番確率の高い単語を選定するため、何度実行しても同じ回答が返ってくる。
  * ハルシネーションの抑止: 逆にアイデア出しなどをさせる場合には0.7などに設定する事がある。

In [64]:
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

## プロンプトの設定

In [65]:
prompt = ChatPromptTemplate.from_template(

"""
あなたは美容・コスメ専門のプロフェッショナルなAIエンジニア兼アドバイザーです。
以下の参考情報（Sephoraのコスメ成分データベース）のみを用いて、ユーザーの質問に答えてください。
提案する際は、ブランド名と商品名、そしてなぜその成分が良いのかを論理的に説明してください。

<参考情報>
{context}
</参考情報>

ユーザーの質問: {input}
"""
)

検索してきたドキュメントの文章だけを綺麗に結合する関数

In [66]:
def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

## ログチェインの構築

ブロックをパイプで繋ぐだけでRAGを構成することができる。

In [67]:
rag_chain = (
    {"context": retriever | format_docs,
     "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## 質問の設定

In [68]:
# 例１
query = "乾燥肌で悩んでいるのですが、保湿成分が含まれたおすすめの化粧水はありますか？"

In [69]:
# 例２
query2 = "ニキビができやすい肌は、なんの成分が必要？"

## 実行

In [71]:
print(f"質問:\n {query}\n")
try:
    response = rag_chain.invoke(query)
    print("AIの回答:\n", response)
except Exception as e:
    print("\nエラーが発生しました。APIキーが正しく設定されているか確認してください。")
    print("詳細:", e)

質問:
 乾燥肌で悩んでいるのですが、保湿成分が含まれたおすすめの化粧水はありますか？

AIの回答:
 提案する化粧水は、DR. JART+のWater Drop Hydrating Moisturizerです。この商品に含まれる保湿成分は、水分を肌にしっかりと閉じ込めることができるヒアルロン酸やグリセリンなどが含まれています。これらの成分は乾燥肌に潤いを与え、肌をしっとりと保湿してくれます。特にWater Drop Hydrating Moisturizerは、水滴のような軽やかなテクスチャーで肌にすっと浸透し、乾燥肌をしっかりとケアしてくれるためおすすめです。
